# Bike Availability Modeling
本 notebook 将流程拆成两部分：
1. 数据清洗（Data Cleaning）
2. 模型训练（Model Training）


## Part A: 数据清洗（Data Cleaning）
读取原始数据，完成时间/天气特征和 lag 特征构造，产出 `df_clean`。


In [33]:
import numpy as np
import pandas as pd
from pathlib import Path

INTERVALS_PER_DAY = 144
INTERVALS_PER_WEEK = 1008

# --- 路径定位（支持在项目根目录或 machine_learning 目录运行）---
cwd = Path.cwd()
if (cwd / "final_merged_data.csv.gz").exists():
    base_dir = cwd
elif (cwd / "machine_learning" / "final_merged_data.csv.gz").exists():
    base_dir = cwd / "machine_learning"
else:
    raise FileNotFoundError("Cannot find final_merged_data.csv.gz in current folder or ./machine_learning")

raw_path = base_dir / "final_merged_data.csv.gz"
print(f"Loading raw data from: {raw_path}")

# --- 读取原始数据 ---
df = pd.read_csv(raw_path)


# --- 时间字段清洗 ---
df['last_reported'] = pd.to_datetime(df['last_reported'], errors='coerce')
df = df.dropna(subset=['last_reported', 'station_id', 'num_bikes_available']).copy()

# --- 时间特征（直接使用原始 day/hour/minute 列）---
for c in ['day', 'hour', 'minute']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.dropna(subset=['day', 'hour', 'minute']).copy()
df['day'] = df['day'].astype(int)
df['hour'] = df['hour'].astype(int)
df['minute'] = df['minute'].astype(int)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)



# --- 温度均值 ---
if 'max_air_temperature_celsius' not in df.columns or 'min_air_temperature_celsius' not in df.columns:
    raise KeyError("Missing required temperature columns: max_air_temperature_celsius, min_air_temperature_celsius")
df['average_temperature_celsius'] = (
    df['max_air_temperature_celsius'] + df['min_air_temperature_celsius']
) / 2.0

# --- 气压均值 ---
if 'max_barometric_pressure_hpa' not in df.columns or 'min_barometric_pressure_hpa' not in df.columns:
    raise KeyError("Missing required pressure columns: max_barometric_pressure_hpa, min_barometric_pressure_hpa")
df['average_pressure_hpa'] = (
    df['max_barometric_pressure_hpa'] + df['min_barometric_pressure_hpa']
) / 2.0

# --- 湿度类别 ---
if 'max_relative_humidity_percent' not in df.columns or 'min_relative_humidity_percent' not in df.columns:
    raise KeyError("Missing required humidity columns: max_relative_humidity_percent, min_relative_humidity_percent")
df['average_humidity_percent'] = (
    df['max_relative_humidity_percent'] + df['min_relative_humidity_percent']
) / 2.0

df['very_humid'] = (df['average_humidity_percent'] >= 90).astype(int)

# --- lag 特征（按站点，shift(1) 防泄漏）---
df = df.sort_values(['station_id', 'last_reported']).reset_index(drop=True)
by_station = df.groupby('station_id')['num_bikes_available']

df['bikes_1d_mean'] = by_station.transform(
    lambda s: s.shift(1).rolling(INTERVALS_PER_DAY, min_periods=INTERVALS_PER_DAY // 2).mean()
)
df['bikes_7d_mean'] = by_station.transform(
    lambda s: s.shift(1).rolling(INTERVALS_PER_WEEK, min_periods=INTERVALS_PER_DAY).mean()
)
df['bikes_7d_std'] = by_station.transform(
    lambda s: s.shift(1).rolling(INTERVALS_PER_WEEK, min_periods=INTERVALS_PER_DAY).std()
)
df['bikes_same_slot_prev_day'] = by_station.shift(INTERVALS_PER_DAY)
df['bikes_same_slot_prev_week'] = by_station.shift(INTERVALS_PER_WEEK)

# has_prev_week: 1=真实历史存在, 0=缺失后将被填补
df['has_prev_week'] = df['bikes_same_slot_prev_week'].notna().astype(int)

# --- 缺失填补（按站点均值，最后回退全局均值）---
station_mean_target = df.groupby('station_id')['num_bikes_available'].transform('mean')
global_mean_target = float(df['num_bikes_available'].mean())
lag_cols = [
    'bikes_1d_mean',
    'bikes_7d_mean',
    'bikes_7d_std',
    'bikes_same_slot_prev_day',
    'bikes_same_slot_prev_week',
]
for col in lag_cols:
    df[col] = df[col].fillna(station_mean_target).fillna(global_mean_target)

# --- 建模字段 ---
features = [
    'station_id',
    'lat',
    'lon',
    'capacity',
    'day',
    'hour_sin',
    'hour_cos',
    'minute',
    'very_humid',
    'average_temperature_celsius',
    'average_pressure_hpa',
    'bikes_1d_mean',
    'bikes_7d_mean',
    'bikes_7d_std',
    'bikes_same_slot_prev_day',
    'bikes_same_slot_prev_week',
    'has_prev_week',
]
target = 'num_bikes_available'

missing_cols = [col for col in features + [target] if col not in df.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

df_clean = df.loc[:, ['last_reported'] + features + [target]].dropna().copy()
df_clean = df_clean.sort_values('last_reported').reset_index(drop=True)
df_clean = df_clean.drop(columns=['last_reported'])

print(f"Cleaned shape: {df_clean.shape}")
print(df_clean[features + [target]].head(3))

# --- 特征类型与基数统计 ---
feature_summary = pd.DataFrame({
    'feature': features + [target],
    'dtype': [str(df_clean[c].dtype) for c in features + [target]],
    'cardinality': [int(df_clean[c].nunique(dropna=False)) for c in features + [target]],
})
print("\n=== Feature Dtype & Cardinality ===")
print(feature_summary.to_string(index=False))

# 低基数特征打印频率分布（避免输出过长）
print("\n=== Frequency (for cardinality <= 20) ===")
for c in features + [target]:
    nun = int(df_clean[c].nunique(dropna=False))
    if nun <= 20:
        print(f"\n[{c}] (nunique={nun})")
        freq = df_clean[c].value_counts(dropna=False, normalize=True).sort_index()
        print((freq * 100).round(2).astype(str) + '%')

# --- 保存清洗后数据 ---
cleaned_path = base_dir / "data_cleaned.csv"
df_clean.to_csv(cleaned_path, index=False) 

print(f"Saved cleaned data: {cleaned_path}")



Loading raw data from: /Users/alex/Desktop/COMP30830-David/machine_learning/final_merged_data.csv.gz
Cleaned shape: (298946, 18)
   station_id        lat       lon  capacity  day  hour_sin  hour_cos  minute  \
0          31  53.350930 -6.265125        20    1       0.0       1.0      10   
1          13  53.336075 -6.252825        30    1       0.0       1.0      10   
2           4  53.346874 -6.272976        20    1       0.0       1.0      10   

   very_humid  average_temperature_celsius  average_pressure_hpa  \
0           0                       13.955               1002.41   
1           0                       13.955               1002.41   
2           0                       13.955               1002.41   

   bikes_1d_mean  bikes_7d_mean  bikes_7d_std  bikes_same_slot_prev_day  \
0      10.652259      10.652259     10.652259                 10.652259   
1      12.959508      12.959508     12.959508                 12.959508   
2      10.550703      10.550703     10.550703   

## Part B: 模型训练（Model Training）
使用 `df_clean` 训练线性回归：
- `station_id` 使用 One-Hot 编码
- 时间顺序切分 70/30
- 输出 CV 与测试集指标并保存模型


In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import joblib
import pickle

X_all = df_clean[features]
y_all = df_clean[target]

categorical_features = ['station_id']
numeric_features = [col for col in features if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('station_ohe', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features),
    ]
)

model = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('regressor', LinearRegression()),
    ]
)

# --- TimeSeries CV ---
cv = TimeSeriesSplit(n_splits=5)
cv_r2_scores = cross_val_score(model, X_all, y_all, cv=cv, scoring='r2')
cv_mae_scores = -cross_val_score(model, X_all, y_all, cv=cv, scoring='neg_mean_absolute_error')
print(f"CV(5)-MAE Mean: {cv_mae_scores.mean():.4f}")
print(f"CV(5)-R² Mean: {cv_r2_scores.mean():.4f}")

# --- 70/30 时间切分 ---
split_idx = int(len(df_clean) * 0.7)
if split_idx == 0 or split_idx == len(df_clean):
    raise ValueError("Not enough data to perform time-based split with 70/30 ratio.")

train_data = df_clean.iloc[:split_idx]
test_data = df_clean.iloc[split_idx:]

X_train = train_data[features]
y_train = train_data[target]
X_test = test_data[features]
y_test = test_data[target]

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Test MAE: {mae:.4f}")
print(f"Test R²: {r2:.4f}")

# --- 系数解释 ---
feature_names = model.named_steps['preprocess'].get_feature_names_out()
coefs = model.named_steps['regressor'].coef_
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
print("Top 20 abs coefficients:")
print(coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=False).index).head(20).to_string(index=False))

# --- 保存模型 ---
joblib_path = base_dir / "bike_availability_model.joblib"
pkl_path = base_dir / "bike_availability_model.pkl"
joblib.dump(model, joblib_path)
with open(pkl_path, "wb") as f:
    pickle.dump(model, f)

print(f"Saved: {joblib_path}")
print(f"Saved: {pkl_path}")


CV(5)-MAE Mean: 5.0185
CV(5)-R² Mean: 0.5439
Test MAE: 3.9067
Test R²: 0.6840
Top 20 abs coefficients:
                    feature  coefficient
 station_ohe__station_id_51    -7.865066
 station_ohe__station_id_38     7.785895
 station_ohe__station_id_92     6.966580
 station_ohe__station_id_98     6.283198
  station_ohe__station_id_1     6.226100
 station_ohe__station_id_70    -6.185982
station_ohe__station_id_105    -5.906156
station_ohe__station_id_103    -5.736799
station_ohe__station_id_111    -5.723764
 station_ohe__station_id_89    -5.661644
 station_ohe__station_id_23     5.168186
 station_ohe__station_id_14     5.108245
 station_ohe__station_id_78    -5.024039
 station_ohe__station_id_52     4.704688
 station_ohe__station_id_79    -4.613080
 station_ohe__station_id_64     4.562550
 station_ohe__station_id_72    -4.462838
station_ohe__station_id_115     4.456445
 station_ohe__station_id_21     4.349821
 station_ohe__station_id_55    -4.328067
Saved: /Users/alex/Desktop/COMP30830